# 画像分類コンペ実践

転移学習・データ拡張・TTA・fold アンサンブルで画像分類コンペに取り組む流れを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/image-classification`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# timm パッケージの確認・自動インストールと、画像コンペ形式のメタデータ CSV を準備します。
import os
import subprocess
import sys

try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import numpy as np
import pandas as pd
import torch

def _ensure_image_comp_data():
    data_dir = os.path.abspath("./.kaggle_input/image-classification")
    os.makedirs(data_dir, exist_ok=True)
    orig_read_csv = pd.read_csv
    if not getattr(orig_read_csv, "_is_img_comp_wrapped", False):
        def _patched_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/<competition>/"):
                rel = filepath_or_buffer[len("/kaggle/input/<competition>/"):]
                filepath_or_buffer = os.path.join(data_dir, rel)
            return orig_read_csv(filepath_or_buffer, *args, **kwargs)
        _patched_read_csv._is_img_comp_wrapped = True
        pd.read_csv = _patched_read_csv

    rng = np.random.default_rng(42)
    labels = rng.choice([0, 1, 2, 3, 4], size=120, p=[0.35, 0.25, 0.20, 0.12, 0.08])
    pd.DataFrame({
        "image_id": [f"img_{i:04d}.jpg" for i in range(len(labels))],
        "label": labels,
    }).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    print(f"Image competition metadata ready at: {data_dir}")

_ensure_image_comp_data()

## 概要

画像分類コンペでは、ImageNet 等で事前学習済みのモデルを使う **転移学習** が標準です。
ゼロから学習するより少データ・短時間で高精度を狙えます。L7（CV）の実践編です。

> **Note:** 画像コンペは GPU が前提です。デバッグは少枚数・小解像度で行い、本学習だけ GPU を有効化して クォータを節約します（L7-09 参照）。

## 1. データ確認

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/<competition>/train.csv")  # image_id, label
df["label"].value_counts()   # クラス分布（不均衡を確認）

クラスの偏り・画像サイズ・破損画像を確認し、層化した 交差検証 を設計します。

## 2. 事前学習モデルのファインチューニング

In [ ]:
# クラス数を訓練メタデータから取得します
n_classes = int(df["label"].nunique())

In [ ]:
import timm, torch.nn as nn

model = timm.create_model("tf_efficientnet_b0", pretrained=True, num_classes=n_classes)
# 通常の PyTorch 学習ループ（AdamW + 小さめ学習率 + 早期終了）

事前学習時より**小さい学習率**で微調整し、検証で早期終了します。

## 3. データ拡張と TTA

In [ ]:
# 学習: 反転・回転・切り抜き・色変換などで水増し（過学習抑制）
# 推論: TTA（複数の拡張版を予測して平均）でスコア底上げ

データ拡張 は画像コンペの効きどころです。正規化の統計量は学習・検証で必ずそろえます。

## 4. fold アンサンブル

各 fold のモデルでテストを予測し、平均します。さらに別アーキテクチャを混ぜると多様性が出ます。

In [ ]:
# 5-fold 学習済みモデルの推論ラッパーとダミー画像バッチ (4枚, RGB, 224x224) を用意してアンサンブル推論を確認します
model.eval()

class _FoldPredictor:
    def __init__(self, net):
        self.net = net
    def predict(self, batch_tensor):
        with torch.no_grad():
            return torch.softmax(self.net(batch_tensor), dim=-1).cpu().numpy()

fold_models = [_FoldPredictor(model) for _ in range(5)]
test = torch.randn(4, 3, 224, 224)

In [ ]:
test_pred = sum(fold_models[k].predict(test) for k in range(5)) / 5

In [ ]:
print("Fold-averaged test_pred shape:", test_pred.shape)
print(test_pred.round(3))

> **Warning:** 検証時の前処理・正規化を推論と一致させること。ここがずれると 交差検証 が当てにならず、 過学習やシェイクアップの原因になります。

## 関連トラック

- L7-04 CV I：CNN 基礎
- L7-05 CV II：転移学習・ファインチューニング
- L7-09 GPU/TPU・メモリ最適化